In [11]:
# CELL 1: Environment, paths, and reproducibility setup
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import util
from llama_cpp import Llama, LlamaGrammar

EMB_DIR = Path("../data/embeddings")
OUTPUT_DIR = Path("../data/results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_FILE = EMB_DIR / "cicids_sample.csv"
EMBED_FILE = EMB_DIR / "cicids_embeddings.npy"

COMMUNITY_FILE = OUTPUT_DIR / "community_assignments.csv"
SUMMARY_FILE = OUTPUT_DIR / "community_summary.csv"
CLUSTER_METRICS_FILE = OUTPUT_DIR / "clustering_metrics.json"
TRIPLES_FILE = OUTPUT_DIR / "community_triples.json"
TRIPLE_METRICS_FILE = OUTPUT_DIR / "triple_metrics.json"

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [12]:
# CELL 2: Load preprocessed sample and embeddings from Stage 2
sample = pd.read_csv(SAMPLE_FILE, index_col="sample_id", low_memory=False)
embeddings = np.load(EMBED_FILE)
assert len(sample) == len(embeddings), "Embedding/CSV length mismatch"
print(f"Loaded {len(sample)} samples | Embedding shape: {embeddings.shape}")

Loaded 10684 samples | Embedding shape: (10684, 384)


In [13]:
# CELL 3: Greedy cosine-threshold clustering (Course method)
communities = util.community_detection(
    embeddings,
    min_community_size=3,
    threshold=0.75,
)
print(f"Found {len(communities)} semantic communities")

Found 18 semantic communities


In [14]:
# CELL 4: Map community indices back to sample rows and save
community_assignments = np.full(len(sample), -1, dtype=int)
for comm_idx, comm_indices in enumerate(communities):
    for idx in comm_indices:
        community_assignments[idx] = comm_idx

valid_mask = community_assignments != -1
community_df = sample[valid_mask].copy()
community_df["community_id"] = community_assignments[valid_mask]
community_df.to_csv(COMMUNITY_FILE, index=False)
print(f"Assigned {len(community_df)} alerts to {community_df['community_id'].nunique()} communities")

Assigned 10683 alerts to 18 communities


In [15]:
# CELL 5: Compute community summary and tactic homogeneity
summary_rows = []
for cid, group in community_df.groupby("community_id"):
    dominant_label = group["Label"].mode().iloc[0]
    dominant_tactic = group["attck_tactic"].mode().iloc[0]
    homogeneity = group["attck_tactic"].value_counts(normalize=True).iloc[0]
    summary_rows.append({
        "community_id": cid,
        "size": len(group),
        "dominant_label": dominant_label,
        "dominant_tactic": dominant_tactic,
        "homogeneity": round(homogeneity, 4)
    })

summary_df = pd.DataFrame(summary_rows).sort_values("size", ascending=False)
summary_df.to_csv(SUMMARY_FILE, index=False)
print(f"Summary saved. {len(summary_df)} communities.")

Summary saved. 18 communities.


In [17]:
# CELL 6: Load local LLM with deterministic parameters (Session 2 spec)
MODEL_PATH = "../models/qwen2.5-3b-instruct-q4_k_m.gguf"
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=2048,
    n_gpu_layers=-1,
    n_threads=8,
    n_batch=512,
    verbose=False,
    seed=RANDOM_SEED,
    temperature=0
)
print("LLM loaded with deterministic parameters")

llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


LLM loaded with deterministic parameters


In [18]:
# CELL 7: Define strict JSON schema and initialize grammar constraint
TRIPLE_SCHEMA = {
    "type": "object",
    "properties": {
        "triples": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "subject": {"type": "string", "enum": ["network_flow"]},
                    "relation": {"type": "string", "enum": ["targets_service", "shows_behavior", "has_duration", "indicates_activity", "communicates_via"]},
                    "target": {"type": "string", "enum": [
                        "ftp_service", "ssh_service", "http_service", "https_service", "dns_service", "unknown_service",
                        "very_short_connection", "long_duration", "high_volume_traffic", "incomplete_handshake",
                        "port_scanning_activity", "brute_force_login_activity", "denial_of_service_behavior",
                        "web_exploitation_activity", "command_and_control_behavior", "data_exfiltration_pattern"
                    ]}
                },
                "required": ["subject", "relation", "target"]
            }
        }
    },
    "required": ["triples"]
}
grammar = LlamaGrammar.from_json_schema(json.dumps(TRIPLE_SCHEMA))

In [19]:
# CELL 8: Prompt builder and extraction function
def extract_triples(text_block):
    prompt = f"""[INST] Extract exactly 4 distinct semantic triples summarizing this alert community.
Choose ONLY from the allowed values. Return valid JSON. No markdown.
ALERTS:
{text_block}
[/INST]"""
    
    out = llm(
        prompt,
        max_tokens=350,
        temperature=0,
        seed=RANDOM_SEED,
        grammar=grammar,
        repeat_penalty=1.2,
        stop=["[/INST]"]
    )
    
    raw = out["choices"][0]["text"].strip()
    try:
        return json.loads(raw).get("triples", [])
    except json.JSONDecodeError:
        objs = re.findall(r'\{[^{}]*"subject"[^{}]*"relation"[^{}]*"target"[^{}]*\}', raw)
        return [json.loads(o) for o in objs if o]

In [20]:
# CELL 9: Warmup and run extraction across all communities
print("Warming up LLM context...")
_ = llm("[INST] Warmup.[/INST]", max_tokens=4, temperature=0, seed=RANDOM_SEED, grammar=grammar)
print("Warmup complete.")

community_triples = {}
failed = []
for cid, group in community_df.groupby("community_id"):
    texts = group["alert_text"].dropna().head(6).tolist()
    block = "\n".join([f"- {t}" for t in texts])
    try:
        community_triples[str(cid)] = extract_triples(block)
    except Exception as e:
        failed.append(cid)
        community_triples[str(cid)] = []

print(f"Extracted triples for {len(community_triples)} communities. Failed: {len(failed)}")

Warming up LLM context...
Warmup complete.
Extracted triples for 18 communities. Failed: 0


In [21]:
# CELL 10: Validate output and save clustering + triple metrics
valid_count = 0
total_count = 0
for triples in community_triples.values():
    for t in triples:
        if all(k in t for k in ["subject", "relation", "target"]):
            valid_count += 1
        total_count += 1

triple_metrics = {
    "n_communities": len(community_triples),
    "coverage": float(len([v for v in community_triples.values() if v]) / max(len(community_triples), 1)),
    "mean_triples_per_community": float(np.mean([len(v) for v in community_triples.values()])),
    "valid_ratio": round(valid_count / max(total_count, 1), 4),
    "failed_communities": failed
}

cluster_metrics = {
    "clustering_method": "sentence_transformers.util.community_detection",
    "embedding_model": "all-MiniLM-L6-v2",
    "threshold": 0.75,
    "min_community_size": 3,
    "n_communities": len(summary_df),
    "mean_homogeneity": float(summary_df["homogeneity"].mean()),
    "random_seed": RANDOM_SEED
}

with open(TRIPLE_METRICS_FILE, "w") as f:
    json.dump(triple_metrics, f, indent=2)
with open(CLUSTER_METRICS_FILE, "w") as f:
    json.dump(cluster_metrics, f, indent=2)
print("Metrics saved.")

Metrics saved.


In [22]:
# CELL 11: Save final triples for Stage 4 knowledge graph construction
with open(TRIPLES_FILE, "w", encoding="utf-8") as f:
    json.dump(community_triples, f, indent=2)
print(f"Triples saved to {TRIPLES_FILE}")

Triples saved to ../data/results/community_triples.json
